### Example 1: Create a Point Column Using Latitude/Longitude

### What is SRID?

SRID stands for Spatial Reference System Identifier.
It is a unique number used to identify the coordinate system and projection method a spatial object uses.

Each SRID defines:

- How coordinates relate to points on Earth
- What units are used (degrees, meters, etc.)
- Whether data is in 2D Cartesian projection or on the real Earth’s curved surface (geoid)

If your geometry has the wrong SRID — spatial functions like distance, area, or intersect can return incorrect results.

In [0]:
from pyspark.sql.functions import expr

# Create DataFrame with store name and coordinates
data = [
    ("Store A", 40.748817, -73.985428),  # Empire State Building
    ("Store B", 51.500729, -0.124625),   # Big Ben
    ("Store C", 35.689487, 139.691706)   # Tokyo City Hall
]

df = spark.createDataFrame(data, ["store_name", "lat", "lon"])

# Add GEOGRAPHY column using ST_POINT
df_geo = df.withColumn("location", expr("ST_POINT(lon, lat)"))

df_geo.show(truncate=False)

### Example 2: Create a GEOMETRY Polygon from WKT
Convert geometries stored as text into spatial objects.

In [0]:
from pyspark.sql.functions import expr

# Sample WKT polygons representing city regions
data = [
    ("Zone 1", "POLYGON((0 0, 4 0, 4 3, 0 3, 0 0))"),
    ("Zone 2", "POLYGON((5 5, 9 5, 9 9, 5 9, 5 5))")
]

df_wkt = spark.createDataFrame(data, ["region_name", "wkt"])

# Convert WKT to GEOMETRY
df_geom = df_wkt.withColumn("boundary", expr("ST_GEOMFROMWKT(wkt)"))

df_geom.show(truncate=False)


This code defines two regions as WKT polygon strings and loads them into a Spark DataFrame.
Using the ST_GEOMFROMWKT() function, it converts the WKT text into a Databricks-native GEOMETRY type, which is required for spatial analysis.
The expr() function lets us use SQL expressions in PySpark to create the boundary column.
Once converted, the geometry can be used for spatial operations like distance calculation, intersections, or area measurement.
Finally, df_geom.show() displays the result, now containing a usable spatial boundary for each region.

### Example 3: Parse GeoJSON to GEOMETRY

GeoJSON is the most commonly used format for spatial data. Let’s load one directly.

In [0]:
from pyspark.sql.functions import expr

# Sample GeoJSON data
data = [
    ("Berlin", """{"type": "Point", "coordinates": [13.4050, 52.5200]}"""),
    ("Paris", """{"type": "Point", "coordinates": [2.3522, 48.8566]}""")
]

df_gj = spark.createDataFrame(data, ["city", "geojson"])

# Create GEOMETRY column using GeoJSON parser
df_parsed = df_gj.withColumn("geometry", expr("ST_GEOMFROMGEOJSON(geojson)")).withColumn("geography", expr("ST_GEOGFROMGEOJSON(geojson)"))


df_parsed.createOrReplaceTempView("df_parsed")
df_parsed.show(truncate=False)


### Save as Delta Lake Table with Spatial Types

We now create a Delta table using our spatial column.

In [0]:
%sql
-- SQL: Save Parsed GeoJSON as Delta
CREATE OR Replace table thedataengineering_00.spatial.cities
USING DELTA
AS SELECT * FROM df_parsed

In [0]:
%sql
select * from thedataengineering_00.spatial.cities

### Run Your First Spatial Query in SQL

Let’s perform a simple calculation — distance between two stores.

In [0]:
%sql
SELECT
  a.city AS city_1,
  b.city AS city_2,
  ST_DISTANCE(
    ST_TRANSFORM(a.geometry, 3857),
    ST_TRANSFORM(b.geometry, 3857)
  ) AS distance_meters
FROM thedataengineering_00.spatial.cities a
CROSS JOIN thedataengineering_00.spatial.cities b
WHERE a.city = 'Berlin' AND b.city = 'Paris';


### What is SRID?

SRID stands for Spatial Reference System Identifier.
It is a unique number used to identify the coordinate system and projection method a spatial object uses.

Each SRID defines:

- How coordinates relate to points on Earth
- What units are used (degrees, meters, etc.)
- Whether data is in 2D Cartesian projection or on the real Earth’s curved surface (geoid)

If your geometry has the wrong SRID — spatial functions like distance, area, or intersect can return incorrect results.

### 🌐 Standard SRIDs in Geospatial Analytics
SRID	Name	Description	Units	Use Case
4326	WGS84 (Lat/Lon)	Earth coordinates in degrees (longitude, latitude)	Degrees	Storing raw geographic data (GPS, GeoJSON, WKT)
3857	Web Mercator	Projected coordinate system used by most web maps	Meters	Calculating real-world distances, buffering, map display
326XX / 327XX	UTM Zones	Accurate for specific longitude bands	Meters	Engineering, surveying within zone
4979	WGS84 (Lat/Lon + elevation)	Adds height dimension	Degrees + meters	3D modeling, drones, aviation

### 🔁 Converting Between SRIDs

When you store geometry in EPSG:4326 (GEOMETRY(4326)), distances are in degrees, not meters.

To compute distances in meters, you convert to a projected CRS like Web Mercator (EPSG:3857):

**ST_TRANSFORM(geometry, 3857)**


This shifts the geometry from lat/lon to a planar representation where:

- X and Y axes are in meters
- Spatial functions like ST_DISTANCE() return correct metric values

📐 Why 4326 and 3857 Are So Important

- 4326 (WGS84) is the standard for storing map data because GPS devices, GeoJSON, and all web mapping APIs (Google, OpenStreetMap) use it for point and polygon definitions.
- 3857 (Web Mercator) is used when you need to perform metric analysis or render on slippy web maps (like OpenLayers, Leaflet, Mapbox).

### In Summary:

- Use 4326 to store and share geospatial data
- Use 3857 (or local UTM SRIDs) to calculate accurate distances/areas in meters
- Always track and preserve SRID info to avoid projection mismatches

In [0]:
from pyspark.sql.functions import expr

# Step 1: Sample GeoJSON data for Pune and Nagpur
data = [
    ("Pune",   """{"type": "Point", "coordinates": [73.8567, 18.5204]}"""),
    ("Nagpur", """{"type": "Point", "coordinates": [79.0882, 21.1458]}""")
]

df_gj = spark.createDataFrame(data, ["city", "geojson"])

# Step 2: Create GEOMETRY and GEOGRAPHY columns from GeoJSON

##Geometry is on a earth model as sphere. Measurements in degrees. Flat geometry.
##Geography is on a on actual earth so distances in meters. Earth based geography.

df_parsed = (
    df_gj
    .withColumn("geometry", expr("ST_GEOMFROMGEOJSON(geojson)"))   # Flat geometry (degrees)
    .withColumn("geography", expr("ST_GEOGFROMGEOJSON(geojson)"))  # Earth-based geography (meters)
)

df_parsed.createOrReplaceTempView("df_cities")
df_parsed.show(truncate=False)

# Step 3: Calculate distance in kilometers using GEOGRAPHY column
distance_query = """
SELECT
  a.city AS city_1,
  b.city AS city_2,
  ST_DISTANCE(ST_TRANSFORM(a.geometry, 3857),ST_TRANSFORM(b.geometry, 3857)) / 1000 AS distance_km
FROM df_cities a
CROSS JOIN df_cities b
WHERE a.city = 'Pune' AND b.city = 'Nagpur'
"""

distance_result = spark.sql(distance_query)
distance_result.show()


### Great-Circle (Aerial) Distance

The great-circle distance (also called geodesic distance) is the shortest distance between two points on the surface of a sphere — like the Earth.
It assumes you're traveling in a straight line “as the crow flies” through the air and uses the Earth’s curvature to compute the shortest possible path.

Used for:

- Flight paths
- High-level geographic calculations
- ST_DISTANCE() with GEOGRAPHY type in Databricks, PostGIS, etc.

Example:
The aerial distance between Pune and Nagpur is ~586 km.

By Road Distance

The road distance considers the actual path you travel by road — via highways, streets, turns, terrain, and real navigation routes.
It is always longer than the aerial distance, because roads are not straight lines and must follow infrastructure and geography.

Used for:

- Real-world logistics (delivery, fleet planning)
- Navigation apps like Google Maps
- Routing algorithms

Example:
The road distance between Pune and Nagpur is ~720 km (varies by route).

### Summary Table
Type	How It's Calculated	Used For	Pune ↔ Nagpur (approx)
Aerial / Great-circle	Shortest path on Earth’s surface	Flight paths, spatial analysis	586 km
Road Distance	Actual road network and traffic rules	Driving, logistics planning	720 km